# From Collecting Data to PCA & ComBat and understanding what I actually did.


# Step 1 - Download data from UCSC Xena

I downloaded Gene Expression Data and Meta Data from UCSC Xena.
Expression Data : RSEM tpm (n=19,131)

* Expression Data : How much genes are expressed per each sample
(gene, sample1, sample2 ... sample(n))
Ex. TP53 12.4 5.2 ... 9.1
    EGFR 1.9  2.2 ... 0.2

Meta Data : TCGA TARGET GTEX selected phenotypes (n=19,131)

* Meta Data : Explain what is each sample
(sample, source, tissue, type)
-To distinguish Expression Data whether sample is from tumor or Healthy Normal (with out this, I can't know whether a sample is Tumor or NAT, or Healthy Normal)
Ex. sample1 TCGA COAD  Tumor
    sample2 GTEx Colon Healthy Normal

Source :
- TCGA COAD (Colon Adenocarcinoma) : Tumor & NAT samples
- GTEx : Healthy Normal colon samples (Sigmoid & Transverse)

Dataset :
- 19,131 samples
- 60,498 genes

What is NAT :
- It's not a cancer tissue, but nearby cancer tissue 
so it can be affected from cancer tissue a lot, but many 
researches defined this as a normal tissue. But I don't 
think it's same with normal tissue far from cancer. 
- And this is why I started this project.
When healthy normal tissue is used as the normal control 
instead of NAT, the selected cancer biomarkers may be different.

* Sigmoid Colon : Flat Colon near Anus
* Transverse Colon : Flat Colon at the top

* RSEM(RNA-Seq by Expectation-Maximization)
: But I got this data from UCSC Xena, so I don't have any idea about this

# Step 1.1 - Load Expression Data & Meta Data

In [7]:
# Code for loading Expression Data & Meta Data
# Before I actually ran this code, I thought the number of genes are 19,131,
# but after I ran this code I noticed that's the number of sample, 19,131 is the number of samples (genes = 60,498), after I noticed this
# I corrected Step 1
# and I understand why should I do this work. not only use claude code
# Because the file was too big, I didn't open. Instead, I used claude code, and it made the biggest mistake

# I used Pandas to load into Python
import pandas as pd

# Load gene expression matrix
expr = pd.read_csv("../data_raw/RSEM tpm (n=19,131)", sep="\t", index_col=0)
# Expression Data : gene expression values for 60,498 genes

# Load sample metadata
meta = pd.read_csv("../data_raw/TCGA TARGET GTEX selected phenotypes (n=19,131).txt", sep="\t", encoding="latin-1")
# Meta Data : shows whether each sample is Tumor, NAT, or Healthy Normal

# Check what we loaded
print("Expression matrix shape:", expr.shape)
print("Metadata shape:", meta.shape)
print("Metadata columns:", meta.columns.tolist())

Expression matrix shape: (60498, 19131)
Metadata shape: (19131, 7)
Metadata columns: ['sample', 'detailed_category', 'primary disease or tissue', '_primary_site', '_sample_type', '_gender', '_study']


By this results, 
I can know that the number of sample is 60,498, and genes are 19,131.
Also, in the Metadata, I can know 19,131 of genes, and 7 types of information which
'sample', 'detailed_category', 'primary disease or tissue', '_primary_site', '_sample_type', '_gender', '_study'

* detailed_category : name of specific disease
* _sample_type : Healthy Normal or Tumor or NAT
* _study : Institute whether data come from TCGA or GTEx

# Step 2 - Filtering COAD-Related Samples

I filtered data, beacause not all 19,131 samples are related COAD, so I filtered only COAD data.

- COAD (Colon Adenocarcinoma) is the most common type of the cancer at Colon
* Colon Cancer = broad concept
* COAD = Specific type of colon cancer 


In [8]:
# To check what values are in the Meta Data, because before filtering, I should know exact string (정확한 문자열을 알아야 함)
# Ex : "COAD," or "Colon Cancer," or "Colon Adenocarcinoma." If I don't know exact string, I can't make a code to filter !

# It shows all exact string at the _sample_type
print("_sample_type values:", meta['_sample_type'].unique())
# Exact strings are "Primary Tumor," "Solid Tissue Normal," "Normal Tissue"
# Do not need other exact strings


# It shows where are datas come from (Institute)
print("\n_study values:", meta['_study'].unique())
# Datas are come from "TCGA," "GTEx."
# Even though, result says also "TARGET," it is related to childhood cancer, so is not needed

# Derive the tissue categorie types of GTEx
print("\ndetailed_category values (GTEx only):", meta[meta['_study'] == 'GTEX']['detailed_category'].unique())
# "Colon - Transverse," "Colon - Sigmoid" are derived
# Others are not needed

# TCGA as well
print("\ndetailed_category values (TCGA Colon only):", meta[meta['detailed_category'].str.contains('Colon', na=False)]['detailed_category'].unique())
# "Colon Adenocarcinoma," "Colon - Transverse," "Colon - Sigmoid" are derived

_sample_type values: ['Primary Tumor' 'Solid Tissue Normal' 'Recurrent Tumor' 'Metastatic'
 'Additional - New Primary' 'Additional Metastatic'
 'Primary Blood Derived Cancer - Peripheral Blood' 'Control Analyte'
 'Cell Line' 'Normal Tissue' 'Recurrent Solid Tumor' 'Primary Solid Tumor'
 'Recurrent Blood Derived Cancer - Bone Marrow'
 'Primary Blood Derived Cancer - Bone Marrow'
 'Post treatment Blood Cancer - Bone Marrow'
 'Post treatment Blood Cancer - Blood'
 'Recurrent Blood Derived Cancer - Peripheral Blood']

_study values: ['TCGA' 'GTEX' 'TARGET']

detailed_category values (GTEx only): ['Cells - Leukemia Cell Line (Cml)' 'Cervix - Endocervix'
 'Cervix - Ectocervix' 'Bladder' 'Fallopian Tube' 'Brain - Amygdala'
 'Brain - Spinal Cord (Cervical C-1)' 'Brain - Hypothalamus'
 'Brain - Putamen (Basal Ganglia)'
 'Brain - Nucleus Accumbens (Basal Ganglia)'
 'Brain - Caudate (Basal Ganglia)' 'Brain - Cerebellar Hemisphere'
 'Brain - Frontal Cortex (Ba9)' 'Brain - Anterior Cingulate Cortex

In [ ]:
# And I can filter COAD data now

# Filter COAD-Related samples only 
coad_meta = meta[
    (
        (meta['detailed_category'] == 'Colon Adenocarcinoma') &
        (meta['_sample_type'].isin(['Primary Tumor', 'Solid Tissue Normal']))
    ) |
    (meta['detailed_category'] == 'Colon - Transverse') |
    (meta['detailed_category'] == 'Colon - Sigmoid')
]
# From "meta" > filtering and move to "coad_meta" if category is
# "Colon Adenocarcinoma", and "Primary Tumor," 
# Or "Colon Adenocarcinoma," and "Solid Tissue Normal"
# Or "Colon - Transverse," 
# Or "Colon - Sigmoid"
# & : and, | : or

print("Filtered samples:", len(coad_meta))
# How many filtered samples 
print(coad_meta['_sample_type'].value_counts())
# Per type("Normal Tissue," "Primary Tumor", "Solid Tissue Normal")
print(coad_meta['detailed_category'].value_counts())
# Per category("Colon Adenocarcinoma," "Colon - Transverse," "Colon - Sigmoid")

Filtered samples: 637
_sample_type
Normal Tissue          308
Primary Tumor          288
Solid Tissue Normal     41
Name: count, dtype: int64
detailed_category
Colon Adenocarcinoma    329
Colon - Transverse      167
Colon - Sigmoid         141
Name: count, dtype: int64
